In [2]:
import h5py
import torch
import numpy as np
import os
import json

import keyrank_rs

import torch.nn as nn

from tqdm import tqdm

In [3]:
if os.name == "nt":
    DATAFOLDER = "C:/Data"
else:
    DATAFOLDER = "/mnt/c/Data"

val_test_hdf = h5py.File(f"{DATAFOLDER}/simpleserial-aes-fix-500-diff.hdf5")

val_test_traces = torch.Tensor(np.array(val_test_hdf['trace']))
val_test_plaintexts = torch.Tensor(np.array(val_test_hdf['data']))
val_test_keys = torch.Tensor(np.array(val_test_hdf['key']))

device = torch.device("cuda")

In [4]:
print(val_test_traces.shape)
print(val_test_plaintexts.shape)
print(val_test_keys.shape)

torch.Size([1000, 500, 5000])
torch.Size([1000, 500, 32])
torch.Size([1000, 16])


In [5]:
def metadata_best_epoch(model_name) -> int:
    with open(f"models/{model_name}/metadata.json") as f:
        metadata = json.load(f)
        val_scores = metadata["scores"][1]
        best_epoch = np.array(val_scores).argmin()
    return best_epoch.item()

def get_traces_mean_std(trace_start, trace_end):
    """Load the mean and std of the training trace set within the given interval"""
    with open(f"misc/standardization/trace{trace_start}_{trace_end}.json") as f:
        info = json.load(f)
        mean = info['training_traces_mean']
        std = info['training_traces_std']

    return mean, std

In [6]:
IMPL = "fixslice"
ARCH = "zhang"
PREDICTION_TARGET = "sbox"
TARGET_BYTE_IDX = [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15
]
TRACE_START = 400
TRACE_END = 1500
SEED = 777

models = []

for target_byte in TARGET_BYTE_IDX:
    model_name = f"{IMPL}-{PREDICTION_TARGET}-byte{target_byte}-{ARCH}-{TRACE_START}_{TRACE_END}-s{SEED}"

    epoch = metadata_best_epoch(model_name)

    model_path = f"models/{model_name}/epoch{epoch}.pt"
    print(model_path)

    model = torch.load(model_path).to(device)

    models.append(model)

models/fixslice-sbox-byte0-zhang-400_1500-s777/epoch14.pt


C:\Users\Ulrik\AppData\Local\Temp\ipykernel_25748\900822681.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_path).to(device)


models/fixslice-sbox-byte1-zhang-400_1500-s777/epoch32.pt
models/fixslice-sbox-byte2-zhang-400_1500-s777/epoch16.pt
models/fixslice-sbox-byte3-zhang-400_1500-s777/epoch12.pt
models/fixslice-sbox-byte4-zhang-400_1500-s777/epoch13.pt
models/fixslice-sbox-byte5-zhang-400_1500-s777/epoch14.pt
models/fixslice-sbox-byte6-zhang-400_1500-s777/epoch17.pt
models/fixslice-sbox-byte7-zhang-400_1500-s777/epoch16.pt
models/fixslice-sbox-byte8-zhang-400_1500-s777/epoch14.pt
models/fixslice-sbox-byte9-zhang-400_1500-s777/epoch13.pt
models/fixslice-sbox-byte10-zhang-400_1500-s777/epoch16.pt
models/fixslice-sbox-byte11-zhang-400_1500-s777/epoch19.pt
models/fixslice-sbox-byte12-zhang-400_1500-s777/epoch7.pt
models/fixslice-sbox-byte13-zhang-400_1500-s777/epoch12.pt
models/fixslice-sbox-byte14-zhang-400_1500-s777/epoch13.pt
models/fixslice-sbox-byte15-zhang-400_1500-s777/epoch14.pt


In [7]:
sample = 259

traces_mean, traces_std = get_traces_mean_std(TRACE_START, TRACE_END)

traces = (val_test_traces[sample, :, TRACE_START:TRACE_END] - traces_mean) / traces_std
plaintexts_B1 = val_test_plaintexts[sample, :, :16] # first plaintext block
plaintexts_B2 = val_test_plaintexts[sample, :, 16:] # second plaintext block
key = val_test_keys[sample]

print(traces.shape)
print(plaintexts_B1.shape)
print(key.shape)

torch.Size([500, 1100])
torch.Size([500, 16])
torch.Size([16])


In [8]:
print("True key:", key.long().tolist())
true_key =  key.long().tolist()

all_sbox_scores = model(traces.to(device))
numpy_scores_ = all_sbox_scores.detach().cpu().numpy()

log_softmax = nn.LogSoftmax(dim=1)

for n_traces in range(2,150):
    guesses = []

    numpy_sbox_scores_slice = numpy_scores_[:n_traces]

    for subkey in range(16):

        plaintext_bytes = plaintexts_B1[:n_traces, subkey]
        plaintext_bytes = plaintext_bytes.long().detach().cpu().numpy().squeeze()

        numpy_keyscores = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_bytes, numpy_sbox_scores_slice)

        x = torch.Tensor(numpy_keyscores)
        x = log_softmax(x)
        x = x.sum(dim=0)
        x = x.argmax(dim=0)

        guesses.append(x.item())

    if true_key == guesses:
        n_traces = n_traces
        print(n_traces,"traces")
        break

print("Full attack:",guesses)    

True key: [66, 59, 237, 182, 158, 214, 87, 231, 242, 124, 155, 22, 62, 118, 10, 48]
68 traces
Full attack: [66, 59, 237, 182, 158, 214, 87, 231, 242, 124, 155, 22, 62, 118, 10, 48]


In [ ]:
"""Compute mean traces needed for full key recovery across N different keys, using both plaintext blocks"""

log_softmax = nn.LogSoftmax(dim=1)



traces_needed = []

for sample_idx in tqdm(range(0,500)):

    traces_ = (val_test_traces[sample_idx, :, TRACE_START:TRACE_END] - traces_mean) / traces_std
    plaintexts_B1_ = val_test_plaintexts[sample_idx, :, :16] # first plaintext block
    plaintexts_B2_ = val_test_plaintexts[sample_idx, :, 16:] # second plaintext block
    true_key_ = val_test_keys[sample_idx].long()


    all_sbox_scores = []
    for model in models:
        sbox_scores = model(traces_.to(device))
        all_sbox_scores.append(sbox_scores.detach().cpu().numpy())

    full_guesses = torch.zeros(500,16)

    for subkey in range(16):
        plaintext_B1_bytes = plaintexts_B1_[:, subkey]
        plaintext_B1_bytes = plaintext_B1_bytes.long().detach().cpu().numpy().squeeze()

        plaintext_B2_bytes = plaintexts_B2_[:, subkey]
        plaintext_B2_bytes = plaintext_B2_bytes.long().detach().cpu().numpy().squeeze()

        all_keyscores = []
        for sbox_scores in all_sbox_scores:
            numpy_keyscores1 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, sbox_scores)
            numpy_keyscores2 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B2_bytes, sbox_scores)

        x = torch.Tensor(numpy_keyscores1 + numpy_keyscores2)
        #x = torch.Tensor(numpy_keyscores2)
        x = log_softmax(x)
        x = x.cumsum(dim=0)
        guesses = x.argmax(dim=1)
        full_guesses[:, subkey] = guesses

    success = (full_guesses.long() == true_key_).all(dim=1)
    n_traces = success.nonzero()[0].item() + 1
    traces_needed.append(n_traces)



mean_traces_needed = np.mean(traces_needed)

mean_traces_needed

100%|██████████| 500/500 [00:35<00:00, 14.22it/s]


29.044

In [ ]:
"""Compute traces needed for 99% accurate full key recovery using 32 outputs from each of 16 models"""

log_softmax = nn.LogSoftmax(dim=1)

traces_needed = []

success_matrix = torch.zeros(500,500)

for sample_idx in tqdm(range(0,500)):

    traces_ = (val_test_traces[sample_idx, :, TRACE_START:TRACE_END] - traces_mean) / traces_std
    plaintexts_B1_ = val_test_plaintexts[sample_idx, :, :16] # first plaintext block
    plaintexts_B2_ = val_test_plaintexts[sample_idx, :, 16:] # second plaintext block
    true_key_ = val_test_keys[sample_idx].long()


    all_sbox_scores = []
    for model in models:
        sbox_scores = model(traces_.to(device))
        all_sbox_scores.append(sbox_scores.detach().cpu().numpy())

    full_guesses = torch.zeros(500,16)

    for subkey in range(16):
        plaintext_B1_bytes = plaintexts_B1_[:, subkey]
        plaintext_B1_bytes = plaintext_B1_bytes.long().detach().cpu().numpy().squeeze()

        plaintext_B2_bytes = plaintexts_B2_[:, subkey]
        plaintext_B2_bytes = plaintext_B2_bytes.long().detach().cpu().numpy().squeeze()

        all_keyscores = []
        for sbox_scores in all_sbox_scores:
            numpy_keyscores1 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, sbox_scores)
            numpy_keyscores2 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B2_bytes, sbox_scores)

            all_keyscores.append(numpy_keyscores1)
            all_keyscores.append(numpy_keyscores2)

        x = torch.Tensor(sum(all_keyscores))
        x = log_softmax(x)
        x = x.cumsum(dim=0)
        guesses = x.argmax(dim=1)
        full_guesses[:, subkey] = guesses

    
    successes = (full_guesses.long() == true_key_).all(dim=1)
    success_matrix[sample_idx, :] = successes


(success_matrix.sum(dim=0) >= 495).nonzero()[0].item()

100%|██████████| 500/500 [08:59<00:00,  1.08s/it]


38

In [10]:
"""Compute traces needed for 99% accurate full key recovery using singular output from each of 16 models

Training subkey == Attack subkey"""

log_softmax = nn.LogSoftmax(dim=1)

traces_needed = []

success_matrix = torch.zeros(500,500)

for sample_idx in tqdm(range(0,500)):

    traces_ = (val_test_traces[sample_idx, :, TRACE_START:TRACE_END] - traces_mean) / traces_std
    plaintexts_B1_ = val_test_plaintexts[sample_idx, :, :16] # first plaintext block
    plaintexts_B2_ = val_test_plaintexts[sample_idx, :, 16:] # second plaintext block
    true_key_ = val_test_keys[sample_idx].long()


    all_sbox_scores = []
    for model in models:
        sbox_scores = model(traces_.to(device))
        all_sbox_scores.append(sbox_scores.detach().cpu().numpy())

    full_guesses = torch.zeros(500,16)

    for subkey in range(16):
        plaintext_B1_bytes = plaintexts_B1_[:, subkey]
        plaintext_B1_bytes = plaintext_B1_bytes.long().detach().cpu().numpy().squeeze()

        plaintext_B2_bytes = plaintexts_B2_[:, subkey]
        plaintext_B2_bytes = plaintext_B2_bytes.long().detach().cpu().numpy().squeeze()

        assert len(models) == 16
        sbox_scores = all_sbox_scores[subkey] # Same subkey for training and attack
        numpy_keyscores1 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, sbox_scores)
        #numpy_keyscores2 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B2_bytes, sbox_scores)


        x = torch.Tensor(numpy_keyscores1)
        x = log_softmax(x)
        x = x.cumsum(dim=0)
        guesses = x.argmax(dim=1)
        full_guesses[:, subkey] = guesses

    
    successes = (full_guesses.long() == true_key_).all(dim=1)
    success_matrix[sample_idx, :] = successes


(success_matrix.sum(dim=0) >= 495).nonzero()[0].item()

100%|██████████| 500/500 [06:34<00:00,  1.27it/s]


196

In [ ]:
"""Compute traces needed for 99% accuracy on individual subkeys using 16 models combined"""


log_softmax = nn.LogSoftmax(dim=1)

# subkey, sample_idx, n_traces
success_matrix = torch.zeros(16,500,500)

for sample_idx in tqdm(range(0,500)):

    traces_ = (val_test_traces[sample_idx, :, TRACE_START:TRACE_END] - traces_mean) / traces_std
    plaintexts_B1_ = val_test_plaintexts[sample_idx, :, :16] # first plaintext block
    plaintexts_B2_ = val_test_plaintexts[sample_idx, :, 16:] # second plaintext block
    true_key_ = val_test_keys[sample_idx].long()

    all_sbox_scores = []
    for model in models:
        sbox_scores = model(traces_.to(device))
        all_sbox_scores.append(sbox_scores.detach().cpu().numpy())

    for subkey in range(16):
        plaintext_B1_bytes = plaintexts_B1_[:, subkey]
        plaintext_B1_bytes = plaintext_B1_bytes.long().detach().cpu().numpy().squeeze()

        plaintext_B2_bytes = plaintexts_B2_[:, subkey]
        plaintext_B2_bytes = plaintext_B2_bytes.long().detach().cpu().numpy().squeeze()

        assert len(models) == 16
        sbox_scores = all_sbox_scores[subkey] # Same subkey for training and attack
        numpy_keyscores1 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, sbox_scores)
        numpy_keyscores2 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B2_bytes, sbox_scores)

        x = torch.Tensor(numpy_keyscores1 + numpy_keyscores2)
        x = log_softmax(x)
        x = x.cumsum(dim=0)
        guesses = x.argmax(dim=1)

        success = (guesses.long() == true_key_[subkey])
        success_matrix[subkey, sample_idx] = success


n_traces_needed = []

for subkey in range(16):
    n_traces_needed.append((success_matrix[subkey].sum(dim=0) >= 495).nonzero()[0].item())

n_traces_needed

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]


NameError: name 'traces_mean' is not defined

In [14]:
print("Subkey: &",  " & ".join([f"${n}$" for n in range(16)]), r"\\")
#print("\\hline")
#print("Mean traces: &", " & ".join([f"${mean:.01f}$" for mean in mean_traces_needed]), r"\\")
print("\\hline")
print("\\makecell[l]{Traces for 99\\% \\\\ accuracy} &", " & ".join([f"${n_traces:.0f}$" for n_traces in n_traces_needed]), r"\\")


#for idx, (mean, n99acc) in enumerate(zip(mean_traces_needed, traces_needed_99acc)):
#    print(f"Subkey {idx:02}, mean: {mean:.03f}, traces needed for 99%: {n99acc}")

Subkey: & $0$ & $1$ & $2$ & $3$ & $4$ & $5$ & $6$ & $7$ & $8$ & $9$ & $10$ & $11$ & $12$ & $13$ & $14$ & $15$ \\
\hline
\makecell[l]{Traces for 99\% \\ accuracy} & $30$ & $24$ & $27$ & $24$ & $53$ & $68$ & $36$ & $107$ & $51$ & $48$ & $46$ & $50$ & $98$ & $77$ & $118$ & $72$ \\


In [65]:
mean_traces_needed = traces_needed.mean(dim=1)

traces_needed_99acc = traces_needed.sort(dim=1)[0][:, -6]

for idx, (mean, n99acc) in enumerate(zip(mean_traces_needed, traces_needed_99acc)):
    print(f"Subkey {idx:02}, mean: {mean:.03f}, traces needed for 99%: {n99acc}")

Subkey 00, mean: 9.732, traces needed for 99%: 30.0
Subkey 01, mean: 5.162, traces needed for 99%: 14.0
Subkey 02, mean: 9.510, traces needed for 99%: 31.0
Subkey 03, mean: 10.574, traces needed for 99%: 33.0
Subkey 04, mean: 19.658, traces needed for 99%: 56.0
Subkey 05, mean: 16.122, traces needed for 99%: 55.0
Subkey 06, mean: 10.902, traces needed for 99%: 34.0
Subkey 07, mean: 10.244, traces needed for 99%: 33.0
Subkey 08, mean: 15.714, traces needed for 99%: 50.0
Subkey 09, mean: 12.196, traces needed for 99%: 39.0
Subkey 10, mean: 13.630, traces needed for 99%: 40.0
Subkey 11, mean: 12.578, traces needed for 99%: 36.0
Subkey 12, mean: 44.284, traces needed for 99%: 152.0
Subkey 13, mean: 23.546, traces needed for 99%: 77.0
Subkey 14, mean: 25.858, traces needed for 99%: 93.0
Subkey 15, mean: 23.010, traces needed for 99%: 71.0


In [ ]:
"""Export data from one attack on a single full key"""


n_traces = 42

true_key =  key.long().tolist()
guesses = []

attack_data_folder = f"misc/attack_data/key{sample}"
os.makedirs(os.path.dirname(attack_data_folder), exist_ok=True)

np.save(f"{attack_data_folder}/sbox_scores.npy", numpy_sbox_scores_slice)
np.save(f"{attack_data_folder}/plaintexts.npy", plaintexts_B1[:n_traces])

for subkey in range(16):

    plaintext_bytes = plaintexts_B1[:n_traces, subkey]
    plaintext_bytes = plaintext_bytes.long().detach().cpu().numpy().squeeze()

    numpy_keyscores = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_bytes, numpy_sbox_scores_slice)

    np.save(f"{attack_data_folder}/keybyte{subkey}_scores.npy", numpy_keyscores)

    x = torch.Tensor(numpy_keyscores).softmax(dim=1)
    x = x.log()
    x = x.sum(dim=0)
    x = x.argmax(dim=0)

    guesses.append(x.item())



print("True key:", true_key)
print("Full attack:",guesses)


attack_info = {
    "implementation" : IMPL,
    "architecture" : ARCH,
    "target_variable" : PREDICTION_TARGET,
    "training_target_byte" : TARGET_BYTE_IDX,
    "trace_interval_start" : TRACE_START,
    "trace_interval_end" : TRACE_END,
    "testing_set_index" : sample,
    "attack_traces" : n_traces,
    "true_key" : true_key,
    "attack_output" : guesses,
}

with open(f"{attack_data_folder}/attack_info.json", 'w') as f:
    json.dump(attack_info, f, indent=4)

True key: [66, 59, 237, 182, 158, 214, 87, 231, 242, 124, 155, 22, 62, 118, 10, 48]
Full attack: [66, 59, 237, 182, 158, 214, 87, 231, 242, 124, 155, 22, 62, 118, 10, 48]
